# Sports Business Operations Exploratory Analysis

**Practice dataset (synthetic, fictional 12-team basketball league, 2022-2025 seasons)**

This notebook explores three linked tables:
- `teams.csv` static team metadata
- `team_season_financials.csv` revenue, payroll, attendance, performance by team/season
- `player_contracts.csv` individual player contracts and performance stats

The goal isn't "who's the best player" it's **business operations questions**:
- Is ticket pricing aligned with demand (attendance %)?
- Which teams are getting the most wins per payroll dollar?
- Which player contracts look like good or bad value relative to performance?
- How does market size relate to revenue and spending?


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

teams = pd.read_csv('teams.csv')
team_season = pd.read_csv('team_season_financials.csv')
players = pd.read_csv('player_contracts.csv')

print(teams.shape, team_season.shape, players.shape)
team_season.head()



(12, 4) (48, 21) (672, 15)


pandas.core.frame.DataFrame

## 1. Revenue mix by team (latest season)

In [ ]:
latest = team_season[team_season['season'] == team_season['season'].max()].copy()

rev_cols = ['ticket_revenue_m', 'sponsorship_revenue_m', 'media_revenue_m', 'merch_revenue_m']
rev_long = latest.melt(id_vars=['team_name'], value_vars=rev_cols,
                        var_name='revenue_type', value_name='amount_m')

fig = px.bar(rev_long, x='team_name', y='amount_m', color='revenue_type',
             title='Revenue Mix by Team (Most Recent Season)',
             labels={'amount_m': 'Revenue ($M)', 'team_name': 'Team'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()


## 2. Are ticket prices aligned with demand?

If a team has high attendance % but a relatively low ticket price, that could signal
room to raise prices. The reverse (low attendance, high price) could signal overpricing.

In [3]:
fig = px.scatter(latest, x='avg_attendance_pct', y='avg_ticket_price',
                  size='total_revenue_m', color='market_size', hover_name='team_name',
                  title='Ticket Price vs. Attendance % (bubble size = total revenue)',
                  labels={'avg_attendance_pct': 'Avg Attendance (% of capacity)',
                          'avg_ticket_price': 'Avg Ticket Price ($)'})
fig.show()


## 3. Wins per payroll dollar spending efficiency

A simple operations metric: how many wins is a team getting per million dollars of payroll?
Lower payroll + high wins = efficient. High payroll + low wins = inefficient.

In [4]:
latest['wins_per_payroll_m'] = (latest['wins'] / latest['payroll_m']).round(3)

eff = latest[['team_name', 'wins', 'payroll_m', 'wins_per_payroll_m']].sort_values(
    'wins_per_payroll_m', ascending=False)

fig = px.bar(eff, x='team_name', y='wins_per_payroll_m',
             title='Wins per $1M Payroll (Spending Efficiency)',
             labels={'wins_per_payroll_m': 'Wins / $M Payroll', 'team_name': 'Team'},
             color='wins_per_payroll_m', color_continuous_scale='RdYlGn')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

eff


,team_name,wins,payroll_m,wins_per_payroll_m
39,Seattle Tide,56,129.26,0.433
40,Phoenix Heat Wave,56,129.47,0.433
44,Portland Pines,57,141.53,0.403
41,Chicago Steel,55,150.49,0.365
38,Denver Peaks,45,128.39,0.350
45,Miami Current,44,129.48,0.340
43,Boston Anchors,50,151.17,0.331
36,St. Louis Comets,43,145.73,0.295
47,Nashville Sound,44,151.21,0.291
37,Austin Rattlers,31,118.21,0.262


## 4. Payroll vs. win percentage does spending buy wins?

A trendline here tells you whether higher payroll is actually correlated with winning,
which is a real front-office question (is the cap space being spent effectively?).

In [5]:
fig = px.scatter(team_season, x='payroll_m', y='win_pct', color='season',
                  hover_name='team_name', trendline='ols',
                  title='Payroll vs. Win % (all seasons)',
                  labels={'payroll_m': 'Payroll ($M)', 'win_pct': 'Win %'})
fig.show()


## 5. Player contract value performance vs. salary

A composite performance score (points + rebounds*0.7 + assists*0.9 + efficiency*0.5)
compared against salary. Players above the trendline are arguably overpaid relative to
output; players below it are good value.

In [6]:
latest_players = players[players['season'] == players['season'].max()].copy()
latest_players['performance_score'] = (
    latest_players['ppg'] * 1.0 +
    latest_players['rpg'] * 0.7 +
    latest_players['apg'] * 0.9 +
    latest_players['per'] * 0.5
)

fig = px.scatter(latest_players, x='performance_score', y='salary_m',
                  hover_name='player_name', hover_data=['team_name', 'position', 'age'],
                  trendline='ols',
                  title='Player Salary vs. Performance Score (Most Recent Season)',
                  labels={'performance_score': 'Performance Score', 'salary_m': 'Salary ($M)'})
fig.show()


## 6. Best contract value by team

For each team, find the player with the best performance-per-dollar useful for a
"who's the most cost-effective roster piece" business angle.

In [7]:
latest_players['value_ratio'] = (latest_players['performance_score'] / latest_players['salary_m']).round(2)

best_value = (latest_players.sort_values('value_ratio', ascending=False)
              .groupby('team_name').head(1)
              .sort_values('value_ratio', ascending=False)
              [['team_name', 'player_name', 'position', 'salary_m', 'performance_score', 'value_ratio']])

fig = px.bar(best_value, x='player_name', y='value_ratio', color='team_name',
             title='Best Contract Value Player per Team',
             labels={'value_ratio': 'Performance per $M Salary', 'player_name': 'Player'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

best_value


,team_name,player_name,position,salary_m,performance_score,value_ratio
657,Minneapolis Frost,Reggie Garrick,SF,0.91,15.75,17.31
668,Nashville Sound,Mason Reyes,SG,0.90,13.12,14.58
516,St. Louis Comets,Cam Lindgren,SF,0.90,11.46,12.73
550,Seattle Tide,Andre Okafor,PF,0.90,10.41,11.57
539,Denver Peaks,Dominic Ellington,SG,2.52,22.00,8.73
581,Chicago Steel,Mason Sutton,C,2.18,14.82,6.80
633,Miami Current,Marcus Boateng,SF,4.97,30.70,6.18
608,Boston Anchors,Dominic Donovan,SF,2.30,13.87,6.03
522,Austin Rattlers,Andre Patterson,SG,2.59,15.47,5.97
591,Atlanta Flux,Dre Vance,C,3.51,19.49,5.55


## 7. Operating income trend by market size

Rolls team-season data up by market size to see if large markets are systematically
more profitable, or if smart operations can close the gap.

In [8]:
market_trend = team_season.groupby(['season', 'market_size'])['operating_income_m'].mean().reset_index()

fig = px.line(market_trend, x='season', y='operating_income_m', color='market_size',
              markers=True,
              title='Avg Operating Income by Market Size, Over Time',
              labels={'operating_income_m': 'Avg Operating Income ($M)', 'season': 'Season'})
fig.show()


## Next steps / ideas to extend this

- Pick ONE question above and write a one-page business recommendation (this is the
  deliverable that goes in the portfolio, not just the charts)
- Try swapping in a real dataset (Spotrac salary data, Kaggle NBA stats) using the
  same notebook structure
- Add a SQL layer: load these CSVs into SQLite and re-write a few of these aggregations
  as SQL queries instead of pandas, to practice both
- Build one of these charts into a simple Power BI dashboard for the "presentation" layer
